# ClusterGCN sa int16 strukturom na ogbn-products (E2)

Ponavlja eksperiment sa tri varijante (osnova, izolovano int64, izolovano int16) na
grafu ogbn-products, K = 500 (najveci klaster mora stati u int16 opseg, ranije merenje
daje 5.045 cvorova), 20 klastera po batch-u, 10 seedova, 30 treninga.

Protokol prilagodjen velicini grafa: 2.500 gradijentnih koraka (100 epoha), izbor
najbolje kontrolne tacke uzorkovanom ocenom na validaciji na svakih 250 koraka
(fanout [15, 10, 5], fiksiran seed po oceni, identican za sve varijante, pa ne kvari
poredjenje identiteta), konacna tacnost tacnim zakljucivanjem po slojevima za najbolju
tacku. Budzet je identican za sve tri varijante, pa su poredjenja unutar grafa cista.

Kontrolni zbirovi prvih 10 batch-eva dokazuju da int64 i int16 varijanta vide identicnu
sekvencu batch-eva. Meri se i energija kartice po treningu.

Procena trajanja: 5 do 7 sati, preskace vec upisane redove.


In [ ]:
SEEDOVI = [42, 123, 7, 1, 2, 3, 4, 5, 6, 8]
K = 500
KLASTERA_PO_BATCHU = 20
VARIJANTE = ['osnova', 'izolovano_int64', 'izolovano_int16']
KORAKA, SLOJEVA, OCENA_SVAKIH = 2500, 3, 250
EVAL_FANOUT = [15, 10, 5]
INFER_BATCH = 4096
INT16_MAKS = 32767


In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging, copy
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import ClusterData, ClusterLoader, NeighborLoader
from torch_geometric.utils import to_undirected, subgraph
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')

REZ = Path('rezultati_int16_products')
REZ.mkdir(parents=True, exist_ok=True)

try:
    import pynvml
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-ml-py'], check=False)
    import pynvml
try:
    pynvml.nvmlInit()
    _ruka = pynvml.nvmlDeviceGetHandleByIndex(0)
    log.info(f'NVML aktivan, trenutna snaga {pynvml.nvmlDeviceGetPowerUsage(_ruka)/1000:.0f} W')
except Exception:
    _ruka = None
    log.info('NVML nedostupan, energija se ne meri')


def energija_j():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetTotalEnergyConsumption(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def snaga_w():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = REZ / f'run_{seed}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(putanja, red):
    novi = not putanja.exists()
    with open(putanja, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(putanja, kolona):
    if not putanja.exists():
        return set()
    with open(putanja) as f:
        return {r[kolona] for r in csv.DictReader(f)}


## Tipovi indeksa, model i koraci treninga

In [ ]:
x_proba = torch.randn(8, 4, device=device)
ei_proba = torch.tensor([[0, 1, 2, 5], [1, 2, 0, 3]], device=device)
conv_proba = SAGEConv(4, 4).to(device)
RACUNSKI_TIP = None
for tip in [torch.int16, torch.int32, torch.int64]:
    try:
        izlaz = conv_proba(x_proba, ei_proba.to(tip))
        izlaz.sum().backward()
        RACUNSKI_TIP = tip
        break
    except (RuntimeError, IndexError, TypeError, NotImplementedError):
        continue
if RACUNSKI_TIP is None:
    RACUNSKI_TIP = torch.int64
log.info(f'najuzi tip za racunanje: {RACUNSKI_TIP}')


In [ ]:
SKRIVENI, LR = 256, 0.01


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    maska = batch.train_mask
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], batch.y[maska])
    gubitak.backward()
    opt.step()


def trening_korak_sastavljen(model, x, y, maska, ei, opt):
    opt.zero_grad()
    out = model(x, ei)
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], y[maska])
    gubitak.backward()
    opt.step()


## Ucitavanje grafa i ocena

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-products', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True
val_idx, test_idx = split_idx['valid'], split_idx['test']
broj_klasa = dataset.num_classes
log.info(f'ogbn-products: {data.num_nodes:,} cvorova, {data.num_edges:,} potega')


In [ ]:
@torch.no_grad()
def oceni_uzorkovano(model, seed, korak):
    model.eval()
    torch.manual_seed(seed * 1000003 + korak)
    loader = NeighborLoader(data, num_neighbors=EVAL_FANOUT, input_nodes=val_idx,
                            batch_size=8192, shuffle=False, num_workers=0)
    tacnih, ukupno = 0, 0
    for b in loader:
        b = b.to(device, non_blocking=True)
        out = model(b.x, b.edge_index)[:b.batch_size]
        tacnih += int((out.argmax(-1) == b.y[:b.batch_size]).sum())
        ukupno += b.batch_size
    return tacnih / ukupno


@torch.no_grad()
def zakljuci_tacno(model):
    model.eval()
    loader = NeighborLoader(data, num_neighbors=[-1], batch_size=INFER_BATCH,
                            shuffle=False, num_workers=8, persistent_workers=False)
    x_sve = data.x
    for i in range(model.slojeva):
        delovi = []
        for b in loader:
            x = x_sve[b.n_id].to(device, non_blocking=True)
            ei = b.edge_index.to(device, non_blocking=True)
            h = model.convs[i](x, ei)[:b.batch_size]
            if i < model.slojeva - 1:
                h = F.relu(model.bns[i](h))
            delovi.append(h.cpu())
            del x, ei, h
        x_sve = torch.cat(delovi)
        torch.cuda.empty_cache()
    pred = x_sve.argmax(-1)
    val = float((pred[val_idx] == data.y[val_idx]).float().mean())
    test = float((pred[test_idx] == data.y[test_idx]).float().mean())
    return val, test


## Particionisanje i priprema klastera

In [ ]:
t0 = time.time()
cd = ClusterData(data, num_parts=K, recursive=False)
metis_vreme = time.time() - t0
klasteri = [kl for kl in ClusterLoader(cd, batch_size=1, shuffle=False)]
najveci = max(kl.num_nodes for kl in klasteri)
if najveci > INT16_MAKS:
    raise ValueError(f'najveci klaster {najveci} ne staje u int16')
potezi64 = [kl.edge_index for kl in klasteri]
potezi16 = [p.to(torch.int16) for p in potezi64]
velicine = [kl.num_nodes for kl in klasteri]
mb64 = sum(p.numel() for p in potezi64) * 8 / 1e6
mb16 = sum(p.numel() for p in potezi16) * 2 / 1e6
mb_globalno = data.edge_index.numel() * 8 / 1e6
log.info(f'K={K}: najveci klaster {najveci}, METIS {metis_vreme:.0f} s, struktura globalno '
         f'{mb_globalno:.0f} MB, izolovano int64 {mb64:.0f} MB, int16 {mb16:.0f} MB')


## Proba ocene pre merenja

In [ ]:
proba = GraphSAGE(data.num_features, SKRIVENI, broj_klasa, SLOJEVA).to(device)
t0 = time.time()
v = oceni_uzorkovano(proba, 0, 0)
log.info(f'proba uzorkovane ocene: {v*100:.1f} % (neobucen model), {time.time()-t0:.1f} s')
t0 = time.time()
pv, pt = zakljuci_tacno(proba)
log.info(f'proba tacnog zakljucivanja: val {pv*100:.1f} %, test {pt*100:.1f} %, {time.time()-t0:.0f} s')
del proba
gc.collect()
torch.cuda.empty_cache()


## Merenje, tri varijante po seedu

In [ ]:
for SEED in SEEDOVI:
    fajl = direktorijum(SEED) / 'int16.csv'
    uradjeno = vec_uradjeno(fajl, 'oznaka')
    for varijanta in VARIJANTE:
        oznaka = f'K{K}_{varijanta}'
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        model = GraphSAGE(data.num_features, SKRIVENI, broj_klasa, SLOJEVA).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        if varijanta == 'osnova':
            loader = ClusterLoader(cd, batch_size=KLASTERA_PO_BATCHU, shuffle=True,
                                   num_workers=8, persistent_workers=True)
            potezi, struktura = None, mb_globalno
        else:
            loader = None
            potezi = potezi64 if varijanta == 'izolovano_int64' else potezi16
            struktura = mb64 if varijanta == 'izolovano_int64' else mb16

        naj_val, naj_korak, najbolji = 0.0, 0, None
        vrh, trening_vreme, korak, epoha = 0.0, 0.0, 0, 0
        cvorova_l, potega_l, snage = [], [], []
        k_cvorova, k_potega, k_zbir = 0, 0, 0
        e0 = energija_j()
        torch.cuda.reset_peak_memory_stats()
        while korak < KORAKA:
            epoha += 1
            model.train()
            grupe = loader if varijanta == 'osnova' else torch.randperm(K).split(KLASTERA_PO_BATCHU)
            for grupa in grupe:
                torch.cuda.synchronize()
                t0 = time.time()
                if varijanta == 'osnova':
                    n_cv, n_po = grupa.num_nodes, grupa.num_edges
                    zbir = int(grupa.edge_index.sum()) if korak < 10 else 0
                    trening_korak(model, grupa, opt)
                else:
                    idx = grupa.tolist()
                    x = torch.cat([klasteri[j].x for j in idx]).to(device, non_blocking=True)
                    y = torch.cat([klasteri[j].y for j in idx]).to(device, non_blocking=True)
                    maska = torch.cat([klasteri[j].train_mask for j in idx]).to(device, non_blocking=True)
                    delovi = [potezi[j] for j in idx]
                    ei = torch.cat(delovi, dim=1).to(device, non_blocking=True).to(RACUNSKI_TIP)
                    brojevi = torch.tensor([p.size(1) for p in delovi], device=device)
                    of, ofs = 0, []
                    for j in idx:
                        ofs.append(of)
                        of += velicine[j]
                    ofseti = torch.tensor(ofs, dtype=RACUNSKI_TIP, device=device)
                    ei = ei + torch.repeat_interleave(ofseti, brojevi)
                    n_cv, n_po = x.size(0), ei.size(1)
                    zbir = int(ei.sum().item()) if korak < 10 else 0
                    trening_korak_sastavljen(model, x, y, maska, ei, opt)
                torch.cuda.synchronize()
                vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                trening_vreme += time.time() - t0
                snage.append(snaga_w())
                if korak < 10:
                    k_cvorova += n_cv
                    k_potega += n_po
                    k_zbir += zbir
                if len(cvorova_l) < 20:
                    cvorova_l.append(n_cv)
                    potega_l.append(n_po)
                korak += 1
                if korak % OCENA_SVAKIH == 0:
                    v = oceni_uzorkovano(model, SEED, korak)
                    if v > naj_val:
                        naj_val, naj_korak = v, korak
                        najbolji = {k: t.cpu().clone() for k, t in model.state_dict().items()}
                    model.train()
                    torch.cuda.reset_peak_memory_stats()
                if korak >= KORAKA:
                    break
        e1 = energija_j()

        model.load_state_dict(najbolji)
        val_tacno, test_tacno = zakljuci_tacno(model)
        upisi_red(fajl, {
            'oznaka': oznaka, 'klastera': K, 'varijanta': varijanta,
            'tip_racunanja': str(RACUNSKI_TIP).replace('torch.', ''),
            'struktura_mb': struktura, 'najveci_klaster': najveci,
            'naj_val_uzorak': naj_val, 'naj_korak': naj_korak,
            'val_tacno': val_tacno, 'test_tacno': test_tacno,
            'ukupno_koraka': korak, 'epoha': epoha, 'metis_s': metis_vreme,
            'cvorova_po_koraku': float(np.mean(cvorova_l)),
            'potega_po_koraku': float(np.mean(potega_l)),
            'vreme_po_koraku_ms': trening_vreme / korak * 1000,
            'memorija_mb': vrh, 'energija_j': e1 - e0,
            'snaga_prosek_w': float(np.nanmean(snage)),
            'snaga_min_w': float(np.nanmin(snage)),
            'snaga_max_w': float(np.nanmax(snage)),
            'kontrola_cvorova': k_cvorova, 'kontrola_potega': k_potega,
            'kontrola_zbir': k_zbir})
        log.info(f'seed {SEED} | {oznaka}: test={test_tacno*100:.2f}% (korak {naj_korak}), '
                 f'{trening_vreme/korak*1000:.0f} ms/korak, vrh {vrh:.0f} MB, '
                 f'struktura {struktura:.0f} MB, {e1-e0:.0f} J')
        del model, opt, najbolji
        if loader is not None:
            del loader
        gc.collect()
        torch.cuda.empty_cache()

log.info('Merenje zavrseno za sve seedove')


## Kontrola identiteta i sazetak

In [ ]:
problemi, razlike = [], []
for SEED in SEEDOVI:
    p = direktorijum(SEED) / 'int16.csv'
    if not p.exists():
        continue
    with open(p) as f:
        redovi = {r['oznaka']: r for r in csv.DictReader(f)}
    a = redovi.get(f'K{K}_izolovano_int64')
    b = redovi.get(f'K{K}_izolovano_int16')
    if a is None or b is None:
        continue
    for kol in ['kontrola_cvorova', 'kontrola_potega', 'kontrola_zbir']:
        if a[kol] != b[kol]:
            problemi.append((SEED, kol, a[kol], b[kol]))
    razlike.append(abs(float(a['test_tacno']) - float(b['test_tacno'])) * 100)
if problemi:
    log.info(f'NEUSPEH, sekvence batch-eva se razlikuju: {problemi}')
elif razlike:
    log.info(f'PROSLO, identicna sekvenca batch-eva u svih {len(razlike)} poredjenja')
    log.info(f'razlika tacnosti int64 naspram int16: najveca {max(razlike):.3f} p.p.')

for varijanta in VARIJANTE:
    t, m, v, e = [], [], [], []
    for SEED in SEEDOVI:
        p = direktorijum(SEED) / 'int16.csv'
        if not p.exists():
            continue
        with open(p) as f:
            redovi = {r['oznaka']: r for r in csv.DictReader(f)}
        r = redovi.get(f'K{K}_{varijanta}')
        if r is None:
            continue
        t.append(float(r['test_tacno']) * 100)
        m.append(float(r['memorija_mb']))
        v.append(float(r['vreme_po_koraku_ms']))
        e.append(float(r['energija_j']))
    if len(t) > 1:
        log.info(f'{varijanta:16s}: test {np.mean(t):.2f} +/- {np.std(t, ddof=1):.2f} %, '
                 f'vrh {np.mean(m):.0f} MB, {np.mean(v):.0f} ms/korak, {np.mean(e):.0f} J')


## Arhiviranje

In [ ]:
import zipfile

arhiva = Path('rezultati_int16_products.zip')
with zipfile.ZipFile(arhiva, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REZ.rglob('*')):
        if p.is_file():
            z.write(p)
log.info(f'{arhiva}: {arhiva.stat().st_size / 1e6:.1f} MB, preuzeti i sacuvati u projekat')
